# 序列并行

- 简单来说，就是把一张 GPU 吃不下的一条超长数据（序列），切成好几段，分给多张 GPU 一起去算。

不过，为了让这个过程在数学上等同于“完整计算”，实际的技术实现比“切分+处理”要稍微精妙一些。我们可以从以下三个层面来深入理解：

1. 为什么要切？（为了解决显存爆炸）

在 Transformer 架构中，<font color='red'>注意力机制（Attention） 的计算复杂度是序列长度的平方级（ $O(N^2)$ ）。</font>
 - 如果不切：当序列长度达到 32K 或 128K 时，存储注意力矩阵和 KV 缓存所需的显存会瞬间撑爆单张显卡（比如 A100）。
 - 切分后：<font color='red'>每张显卡只需要存储和处理序列的一小部分</font>（比如 1/P），显存压力直接降低为原来的 1/P。
 
2. 怎么切？（两种主流流派）

虽然都是“拆分”，但在具体怎么拆分和协作上，主要有两种不同的技术路线：

 - 流派一：DeepSpeed Ulysses（维度转换法）
    - 切法：把序列切成 P 份，每张卡拿一份。
    - 处理：通过一种叫 All-to-All 的通信方式，把“序列维度”的信息转换到“注意力头维度”上。
    - 结果：每张卡虽然只算一部分序列，但它负责计算所有的“注意力头”。算完后再转回来。
    - 特点：通信量小，但受限于注意力头的数量（卡数不能超过头数）。
    
 - 流派二：Ring-Attention（环形接力法）
    - 切法：同样把序列切成 P 份。
    - 处理：<font color='red'>显卡们连成一个环。每张卡算自己这一块序列时，不够的信息（其他卡的 Key/Value）通过 P2P 通信像接力棒一样传过来。</font>
    - 结果：通过多轮“接收-计算-发送”，每张卡最终都能拼凑出全局的上下文信息。
    - 特点：显存利用率极致，没有卡数限制，非常适合超长文本（如 1M 上下文）。
    
    
3. 代价是什么？（用通信换显存）

正如你所说，我们是“一同处理”。但这个“一同”是有代价的。
- 通信开销：<font color='red'>因为每张卡手里只有一半的拼图，它们必须频繁地互相交换数据（通信），才能拼出完整的画面。</font>
- 核心权衡：序列并行本质上是用网络通信时间来换取显存空间。只要通信够快（比如用 NVLink），这个交换就是非常划算的。


# 序列并行（Sequence Parallelism, SP）
- 的核心思想确实是将输入序列在“序列维度”上进行切分，让不同的计算设备（如GPU）各自处理序列的一部分，从而降低单卡显存压力并提升长序列训练效率。

## 一、为什么需要序列并行？
- 在大模型训练中，尤其是处理超长上下文（如32K token）时，中间激活值（Activations）的显存占用会急剧膨胀。
- 即使使用了张量并行（TP），像 LayerNorm、Dropout 这类操作仍需在完整序列上进行，导致显存瓶颈。
- <font color='red'>序列并行正是为了解决这一问题而设计，它专门针对这些“无法被张量并行切分”的操作进行优化。</font>
- 限制：通常只在TP组内部进行，不独立扩展

## 二、序列并行如何切分？
- 切分对象：
  - <font color='red'>不是模型权重，而是激活值（Activations）</font>，特别是 Transformer 层中 LayerNorm 和 Dropout 的输出。
- 切分维度：
  - 沿着序列长度维度（sequence dimension） 切分。例如，一个形状为 [s, b, h] 的张量（s=序列长度，b=batch size，h=隐藏层维度），会被切分成多个子张量，每个 GPU 只持有其中一段序列。
- 执行流程：
  - 前向传播时，<font color='red'>每个 GPU 独立计算自己负责的子序列上的 LayerNorm 和 Dropout。</font>
  - 随后，通过 All-Gather 或 Ring-Attention 等通信原语，将各 GPU 的结果拼接回完整序列，供后续的注意力机制或 MLP 层使用。
  - 反向传播时，梯度也按相同方式切分和聚合。
  
## 三、序列并行与张量并行的关系
- 序列并行通常作为张量并行的补充策略，而非替代。
- 它不改变模型权重的分布，只在特定层（如 LayerNorm）对激活值做序列维度的切分。
- 这种组合策略能同时降低权重存储和激活值存储的显存开销，<font color='red'>是训练超长序列模型的标配方案。</font>

## 四、主流实现方式
- DeepSpeed-Ulysses：通过 All2All 通信高效分发 Q/K/V/O 张量片段。
- <font color='red'>Ring-Attention：采用环形通信拓扑，减少通信次数，适合超长序列。</font>
- USP (Unified Sequence Parallel)：<font color='red'>融合上述两种方法</font>，提出“2D混合序列并行”，在 LLM 和 DiT 场景中性能提升显著。
## 五、适用场景
- 序列长度超过 8K token，且显存接近上限。
- <font color='red'>希望在保持模型结构不变的前提下，进一步压缩激活值显存。</font>
- <font color='green'>配合张量并行使用，实现更极致的显存优化。</font>

# 举例介绍

假设我们正在训练一个模型，当前的输入数据形状是 [序列长度=8, 隐藏层维度=4]。为了方便演示，我们把它简化为一个 8行4列 的矩阵。

## 🎯 场景设定
- 总序列长度 (Sequence Length)：8 个 token。
- 隐藏层维度 (Hidden Dim)：4。
- GPU 数量：2 张卡（GPU 0 和 GPU 1）。
- 当前操作：LayerNorm（层归一化）。

## 🚫 没有序列并行时（传统做法）
<font color='red'>LayerNorm 需要计算整个序列的均值和方差。因此，每张 GPU 上都必须保存完整的 8×4 矩阵。</font>
- GPU 0 显存占用：8 × 4 = 32 个单位。
- GPU 1 显存占用：8 × 4 = 32 个单位。
- 痛点：如果序列长度是 32K，这个 8×4 就会变成 32000×4，单卡显存瞬间爆炸。

## ✨ 使用序列并行后（切分激活值）
现在，系统沿着“序列长度”这个维度，把数据一刀切成两半：
- GPU 0：只负责处理前 4 个 token。它显存里只存一个 4×4 的矩阵（即原矩阵的第 1-4 行）。
- GPU 1：只负责处理后 4 个 token。它显存里也只存一个 4×4 的矩阵（即原矩阵的第 5-8 行）。
此时，显存占用直接减半！ 每张卡只需要存 16 个单位。

## ⚙️ 接下来怎么计算？（核心难点）
你可能会问：LayerNorm 不是需要算整个序列的均值吗？现在数据被切开了，怎么算？
这就是序列并行的精妙之处。它把一次大计算拆成了三步：
1. 局部计算 (Local Compute)：
- <font color='red'>GPU 0 和 GPU 1 各自在本地计算自己那 4 个 token 的局部均值和方差。</font>
2. 通信同步 (Communication)：
- 两张卡通过通信（比如 All-Gather 或 All-Reduce），<font color='red'>把各自的“局部统计量”交换一下，然后在内存里快速汇总，</font>得出整个 8 个 token 的全局均值和方差。
3. 本地归一化 (Local Normalize)：
- <font color='red'>拿到全局均值和方差后，GPU 0 和 GPU 1 再次在本地，用这个全局统计量去归一化自己手里的那 4 个 token。</font>

## 🔄 后续流程
- 经过 LayerNorm 后，数据依然是切分状态（各 4 个 token）。接着，系统会发起一次 All-Gather 通信，<font color='red'>把 GPU 0 的前半段和 GPU 1 的后半段重新拼接起来，恢复成完整的 8×4 矩阵，然后再交给下一层的 Attention（注意力机制）去处理。</font>

## 总结这个例子：
- 序列并行就像是把一个巨大的“拼图任务”分给两个人。每个人只负责拼自己手里的那一半，但在需要看全局（算均值）的时候，两个人互相通个气（通信），然后再各自完成自己那一半的上色（归一化）。这样既省了桌子空间（显存），又提高了效率。

# 核心矛盾：为什么它们不能像 MLP 那样切分？

- 在标准的张量并行中，大矩阵乘法（如 MLP 层的 Y=XW ）是可以完美切分的。比如把权重矩阵按列切开，GPU0 算一半输出，GPU1 算另一半输出，最后拼起来就行。<font color='green'>在这个过程中，每个 GPU 不需要看到完整的输入序列 X ，只需要自己负责的那部分输入即可。</font>

- <font color='red'>但是，LayerNorm 和 Dropout 打破了这个规则：</font>

## LayerNorm 的“全局依赖”
- 原理：LayerNorm 需要对输入张量的特定维度计算均值 μ 和方差 σ^2。
- 冲突：<font color='green'>在 Transformer 中，LayerNorm 通常作用在隐藏层维度上，但它的统计量计算往往依赖于整个序列或者至少是当前样本的所有 Token。如果输入序列被切分了（比如 GPU0 只有前一半 Token），它就无法独立计算出正确的全局均值和方差。</font>
- 后果：<font color='red'>如果不做特殊处理，每张卡都必须通过通信拿到完整的序列才能算 LayerNorm，这就导致显存并没有节省下来。</font>

## Dropout 的“随机性陷阱”
- 原理：Dropout 会随机将一部分神经元置零。
- 冲突：为了保证训练的正确性，所有 GPU 必须使用完全相同的随机掩码。如果 GPU0 丢弃了第 1 个 Token，而 GPU1 保留了第 1 个 Token（因为它以为那是第 5 个），模型就乱了。
- 后果：<font color='green'>传统做法要求所有 GPU 同步随机种子并生成相同的完整掩码，这同样限制了并行效率。</font>

## 序列并行的解决方案：切分激活值
为了解决上述问题，序列并行引入了一个新的策略：既然权重没法切，那我就切中间产生的激活值。
具体流程如下：
1. 前向传播开始时：输入数据还是完整的。
2. 遇到 LayerNorm/Dropout 时：
  - <font color='green'>系统不再让所有 GPU 都保留完整序列。</font>
  - 而是通过通信（如 All-Gather），将完整的序列分发出去，然后立即在本地进行切分。
  - 关键点：LayerNorm 的计算被重构了。
    - 它先在各卡上计算局部的统计量，然后通过 All-Reduce 汇总全局统计量，
    - 最后各卡利用全局统计量去归一化自己手里的那一小块序列。
3. 计算结束后：
  - 此时，GPU0 手里只有序列的前半段激活值，GPU1 只有后半段。
  - 显存节省达成！ 原本需要存 8×4 的激活值，现在每张卡只存 4×4 。
4. 后续计算：
  - <font color='green'>当这些被切分的激活值进入下一个线性层（MLP）时，由于线性层本身支持张量并行，这种“残缺”的输入正好可以直接使用，无需再次拼接。</font>

## 总结
所以，你的理解完全正确。<font color='green'>序列并行本质上是一种针对非矩阵乘法操作的显存优化技术。</font>
- 它处理的对象：<font color='red'>确实是 LayerNorm、Dropout 以及随后的残差连接产生的中间激活值。</font>
- 它的手段：<font color='blue'>将这些激活值在序列维度上切分，分散存储到不同 GPU 上。</font>
- 它的收益：将这部分显存占用从 O(S) 降低到 O(S/N) （ N 为 GPU 数量），使得训练超长序列成为可能。

# 序列并行和上下文并行之间的区别是？

- 序列并行（Sequence Parallelism, SP）和上下文并行（Context Parallelism, CP）虽然都是为了解决大模型训练中的显存瓶颈，<font color='blue'>并且都是沿着“序列维度”对数据进行切分，但它们在切分范围、核心目的以及通信机制上有着本质的区别。</font>

- 简单来说，<font color='red'>序列并行（SP）是张量并行的“辅助补丁”，而上下文并行（CP）则是为了解决超长序列而诞生的“终极武器”。</font>

以下是它们的核心区别：
## 1. 切分范围与适用模块
- 序列并行 (SP)：
  - <font color='red'>切分是局部的</font>。它只针对 Transformer 中那些“无法被张量并行（TP）切分”的模块，比如 LayerNorm 和 Dropout。对于核心的 Self-Attention（自注意力）机制，SP 并不切分序列，而是通过 All-Gather 通信将序列拼回完整状态后再进行计算。
- 上下文并行 (CP)：
  - <font color='red'>切分是全局的</font>。它将整个网络输入和所有中间激活值都沿序列维度进行划分，包括最核心的注意力计算（Attention）。每个 GPU 从头到尾拿到的都是分区后的子序列。

## 2. 核心目的与能力上限
- 序列并行 (SP)：
  - 目的是减少激活值显存冗余。它是张量并行的补充，配合 TP 使用，能把序列长度撑长一些（比如几万 token），但无法从根本上解决注意力机制带来的显存爆炸问题。
- 上下文并行 (CP)：
  - 目的是彻底打破输入序列长度的物理限制。因为注意力机制的显存占用随序列长度呈二次方增长，当序列长度超过 128K 甚至达到百万级时，只有 CP 才能通过分摊激活值内存，让超长上下文的训练成为可能。

## 3. 通信模式
- 序列并行 (SP)：
  - 主要使用全局集合通信。在进入 Attention 前使用 All-Gather 拼接序列，在计算完 Attention 后使用 Reduce-Scatter 重新切分序列。
- 上下文并行 (CP)：
  - 主要使用点对点环形通信（Ring Attention）。因为每个 Token 都需要“看”序列里所有的 Token，CP 将 GPU 组织成环形拓扑，让 K/V 张量在 GPU 之间像接力棒一样流动。这种设计允许通信与计算重叠，隐藏了通信延迟。

## 4. 直观对比总结

| 维度       | 上下文并行（CP）                        | 序列并行（SP）                       |
| -------- | -------------------------------- | ------------------------------ |
| **切分对象** | 输入序列（token序列）                    | 序列维度上的激活值/中间状态                 |
| **主要目标** | 解决**长序列无法放入单卡显存**的问题             | 解决**注意力计算中的内存瓶颈**（特别是激活值）      |
| **典型应用** | Ring Attention、DeepSpeed Ulysses | Megatron-LM 的序列并行              |
| **通信模式** | 通常采用环形通信（Ring）传递KV Cache         | All-Gather/Reduce-Scatter 集合通信 |
| **计算特点** | 各卡只计算本地序列块的注意力                   | 将序列维度切分到TP组内，配合张量并行            |


## 总结：
- 如果你在处理几万长度的文本，使用 TP + SP 组合就能很好地平衡计算与显存；
- <font color='red'>但如果你要训练几十万甚至上百万 Token 的超长上下文模型，就必须引入 CP（上下文并行） 来彻底瓦解注意力机制的显存瓶颈。</font>

## 联系与演进
###  目标一致：
  - 都是为了支持更长序列的训练，解决显存瓶颈
###  互补关系：
  - 在 Megatron-LM 的框架中，<font color='red'>SP 是 TP 的补充</font>，处理 TP 未覆盖的序列维度
  - CP 是更独立的维度，可以与 TP、SP、DP、PP 正交组合
###  现代融合趋势：
  - 一些新框架（如 DeepSpeed Ulysses、Megatron 的新版本）模糊了界限
  - DeepSpeed Ulysses 既做序列切分又做all-to-all通信，兼具两者特点
  - 现在业内有时会把"序列并行"作为广义术语，包含上下文并行 
###  组合使用：

典型配置：(DP + TP + PP + SP/CP)
- TP：切分注意力头和FFN（8卡）
- SP：<font color='red'>在TP组内</font>切分序列维度的激活（减少LN显存）
- CP：<font color='red'>跨TP组</font>进一步切分超长序列（处理百万token）
- PP：切分层
- DP：数据并行


## 简单类比
- CP 像是把一本很长的书分章节给不同的人读，每个人需要知道其他章节的关键信息（通过通信获取）
- SP 像是把同一页纸竖着裁开，每个人读一列，但需要在同一批人内部拼起来才能理解整页